# 极客项目 ch06：ASCII 文本图形

对应《Python Playground》第 6 章（ASCII Art）。原书 argparse/main 改为演示 cell：程序合成一张测试图转成 ASCII 并打印，输出文件写入系统临时目录（seed=2029）。

In [1]:
# ===== 公共设置 =====
import os
import random
import tempfile
import numpy as np
from PIL import Image

random.seed(2029)
np.random.seed(2029)

## 书 6.3.1 灰度梯度

70 级 `gscale1` / 10 级 `gscale2`，字符由暗到亮排布。

In [2]:
# ===== 书 6.3.1 节 两种灰度梯度 =====
# 70 levels of gray
gscale1 = "$@B%8&WM#*oahkbdpqwmZO0QLCJUYXzcvunxrjft/\\|()1{}[]?-_+~<>i!lI;:,\"^`'. "
# 10 levels of gray
gscale2 = '@%#*+=-:. '


## 书 6.3.2 计算平均亮度

tile 转灰度 numpy 数组后取平均。

In [3]:
# ===== 书 6.3.2 节 getAverageL() =====
def getAverageL(image):
    """Given PIL Image, return average value of grayscale value"""
    im = np.array(image)
    w, h = im.shape
    return np.average(im.reshape(w*h))


## 书 6.3.3 从图像生成 ASCII 内容

按 cols 切列、aspect ratio×scale 定行高，逐 tile 取平均亮度查表。

In [4]:
# ===== 书 6.3.3 节 covertImageToAscii() =====
def covertImageToAscii(fileName, cols, scale, moreLevels):
    """Given Image and dims (rows, cols) returns an m*n list of Images
    （函数名 covertImageToAscii 为原书如此）"""
    global gscale1, gscale2
    # open image and convert to grayscale
    image = Image.open(fileName).convert('L')
    # store dimensions
    W, H = image.size[0], image.size[1]
    print("input image dims: %d x %d" % (W, H))
    # compute width of tile (浮点除法避免截断误差，原书如此)
    w = W/cols
    # compute tile height based on aspect ratio and scale
    h = w/scale
    # compute number of rows
    rows = int(H/h)

    print("cols: %d, rows: %d" % (cols, rows))
    print("tile dims: %d x %d" % (w, h))

    # check if image size is too small
    if cols > W or rows > H:
        print("Image too small for specified cols!")
        exit(0)

    # ascii image is a list of character strings
    aimg = []
    for j in range(rows):
        y1 = int(j*h)
        y2 = int((j+1)*h)
        # correct last tile
        if j == rows-1:
            y2 = H
        # append an empty string
        aimg.append("")
        for i in range(cols):
            x1 = int(i*w)
            x2 = int((i+1)*w)
            # correct last tile
            if i == cols-1:
                x2 = W
            # crop image to extract tile
            img = image.crop((x1, y1, x2, y2))
            # get average luminance
            avg = int(getAverageL(img))
            # look up ascii char
            if moreLevels:
                gsval = gscale1[int((avg*69)/255)]
            else:
                gsval = gscale2[int((avg*9)/255)]
            # append ascii char to string
            aimg[j] += gsval

    return aimg


## 演示：合成图像 → ASCII 文本图形

400×300 测试图（水平渐变 + 暗圆 + 亮方块）写入临时目录；cols=80、scale=0.43、70 级灰度；书 6.3.5 的写文本文件一步一并演示。

In [5]:
# ===== 演示：生成测试图 → ASCII 文本图形（书 6.3.5 写文件并入此 cell）=====
from PIL import ImageDraw

work = tempfile.mkdtemp(prefix='pg_ch06_')
demo_png = os.path.join(work, 'demo.png')

W, H = 400, 300
img = Image.new('L', (W, H))
dr = ImageDraw.Draw(img)
for x in range(W):                            # 水平渐变背景
    dr.line([(x, 0), (x, H)], fill=int(255*x/W))
dr.ellipse((120, 70, 280, 230), fill=40)      # 暗圆
dr.rectangle((40, 200, 140, 280), fill=230)   # 亮方块
img.save(demo_png)

aimg = covertImageToAscii(demo_png, cols=80, scale=0.43, moreLevels=True)

out_path = os.path.join(work, 'out.txt')      # 书 6.3.5
with open(out_path, 'w', encoding='utf-8') as f:
    for row in aimg:
        f.write(row + '\n')
print("ASCII art written to %s" % out_path)
for row in aimg:
    print(row)

input image dims: 400 x 300
cols: 80, rows: 25
tile dims: 5 x 11
ASCII art written to C:\Users\17272\AppData\Local\Temp\pg_ch06_eetgkss5\out.txt
$@@B%8&WMM#*oahhkbdpqwmZZO0QLCCJUYXzccuunxrjfft/\|())1{}[]??-_+~<>ii!lI;:,,"^`'.
$@@B%8&WMM#*oahhkbdpqwmZZO0QLCCJUYXzccuunxrjfft/\|())1{}[]??-_+~<>ii!lI;:,,"^`'.
$@@B%8&WMM#*oahhkbdpqwmZZO0QLCCJUYXzccuunxrjfft/\|())1{}[]??-_+~<>ii!lI;:,,"^`'.
$@@B%8&WMM#*oahhkbdpqwmZZO0QLCCJUYXzccuunxrjfft/\|())1{}[]??-_+~<>ii!lI;:,,"^`'.
$@@B%8&WMM#*oahhkbdpqwmZZO0QLCCJUYXzccuunxrjfft/\|())1{}[]??-_+~<>ii!lI;:,,"^`'.
$@@B%8&WMM#*oahhkbdpqwmZZO0QLCCJUYXzccuunxrjfft/\|())1{}[]??-_+~<>ii!lI;:,,"^`'.
$@@B%8&WMM#*oahhkbdpqwmZZO0QLCCJCQZqpbkhhkbpZQXr\|())1{}[]??-_+~<>ii!lI;:,,"^`'.
$@@B%8&WMM#*oahhkbdpqwmZZO0QLOqhooooooooooooooook0u()1{}[]??-_+~<>ii!lI;:,,"^`'.
$@@B%8&WMM#*oahhkbdpqwmZZO0mhoooooooooooooooooooooobz1{}[]??-_+~<>ii!lI;:,,"^`'.
$@@B%8&WMM#*oahhkbdpqwmZZZbooooooooooooooooooooooooooq\}[]??-_+~<>ii!lI;:,,"^`'.
$@@B%8&WMM#*oahhkbdpqwmZZhooooooooooooooooooo